# Decoding analysis — sub-pil01 ses-1

Analyses results of both decoding scripts after the timing fix.

## Contents
1. **Gabor orientation decoding** — Von Mises basis-set model, circular posterior
2. **Abstract value (CHF) decoding** — Gaussian nPRF model, scalar posterior

For each analysis we compare all available variants (mask × noise model × smoothing)
and show detailed plots for the best variant.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from abstract_values.utils.data import BIDS_FOLDER

SUBJECT = 'pil01'
SESSION = 1
ses_label = f'ses-{SESSION}'

sns.set_style('ticks')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

---
## 1 · Gabor orientation decoding

In [ ]:
# ── helper functions (orientation, π-periodic) ────────────────────────────────

def circular_mean_posterior(pdfs):
    """π-periodic circular mean from posterior DataFrame.

    pdfs : DataFrame (n_trials × n_orientations), columns = orientations in rad.
    Returns dict with decoded_rad, concentration, circ_std_deg, z.
    """
    orientations = pdfs.columns.values.astype(float)
    posterior    = pdfs.values
    norm_factor  = np.trapz(posterior, orientations, axis=1)[:, np.newaxis]
    post_norm    = posterior / np.where(norm_factor > 0, norm_factor, 1.0)
    z = np.trapz(post_norm * np.exp(2j * orientations[np.newaxis, :]),
                 orientations, axis=1)
    decoded_rad   = (np.angle(z) / 2) % np.pi
    concentration = np.abs(z)
    circ_std_deg  = np.rad2deg(np.sqrt(-2 * np.log(concentration.clip(1e-9))))
    return dict(decoded_rad=decoded_rad, concentration=concentration,
                circ_std_deg=circ_std_deg, z=z)


def wrap_pi_half(x):
    """Wrap angle to [−π/2, π/2]."""
    return (x + np.pi / 2) % np.pi - np.pi / 2


def circular_correlation(x, y):
    """Circular-circular correlation for π-periodic data (Jammalamadaka-SenGupta)."""
    x2, y2 = 2 * x, 2 * y
    mu_x2  = np.arctan2(np.mean(np.sin(x2)), np.mean(np.cos(x2)))
    mu_y2  = np.arctan2(np.mean(np.sin(y2)), np.mean(np.cos(y2)))
    dx, dy = np.sin(x2 - mu_x2), np.sin(y2 - mu_y2)
    denom  = np.sqrt(np.sum(dx**2) * np.sum(dy**2))
    return float(np.sum(dx * dy) / denom) if denom > 0 else np.nan

In [ ]:
# ── discover + load all gabor decoding variants ───────────────────────────────
gabor_dir = (Path(BIDS_FOLDER) / 'derivatives' / 'decoding' / 'gabor'
             / f'sub-{SUBJECT}' / ses_label / 'func')

gabor_files = sorted(gabor_dir.glob('*_pars.tsv'))
print(f'Found {len(gabor_files)} gabor variant(s):')
for f in gabor_files:
    print(' ', f.name)

gabor_results = {}
for tsv in gabor_files:
    pdfs = pd.read_csv(tsv, sep='\t',
                       index_col=['session', 'run', 'trial_nr', 'true_orientation_rad'])
    pdfs.columns = pdfs.columns.astype(float)

    stats_ = circular_mean_posterior(pdfs)
    true_rad    = pdfs.index.get_level_values('true_orientation_rad').values
    decoded_rad = stats_['decoded_rad']
    error_rad   = wrap_pi_half(decoded_rad - true_rad)
    error_deg   = np.rad2deg(error_rad)
    circ_corr   = circular_correlation(true_rad, decoded_rad)
    mae         = np.mean(np.abs(error_deg))
    median_ae   = np.median(np.abs(error_deg))

    res = pd.DataFrame({
        'true_rad':      true_rad,
        'decoded_rad':   decoded_rad,
        'true_deg':      np.rad2deg(true_rad),
        'decoded_deg':   np.rad2deg(decoded_rad),
        'error_rad':     error_rad,
        'error_deg':     error_deg,
        'abs_error_deg': np.abs(error_deg),
        'circ_std_deg':  stats_['circ_std_deg'],
        'concentration': stats_['concentration'],
    }, index=pdfs.index)

    gabor_results[tsv.stem] = dict(
        pdfs=pdfs, results=res,
        mae=mae, median_ae=median_ae, circ_corr=circ_corr)
    print(f'  {tsv.stem}:')
    print(f'    MAE={mae:.1f}°  median_AE={median_ae:.1f}°  circ_r={circ_corr:.3f}  (chance≈45°)')

In [ ]:
# ── comparison table + bar chart ──────────────────────────────────────────────
gabor_summary = pd.DataFrame([
    dict(variant=k, MAE_deg=v['mae'], median_AE_deg=v['median_ae'],
         circ_r=v['circ_corr'])
    for k, v in gabor_results.items()
]).sort_values('MAE_deg')

display(gabor_summary.set_index('variant').round(2))

labels = [k.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
          for k in gabor_summary['variant']]

fig, axes = plt.subplots(1, 3, figsize=(14, max(3, len(labels) * 0.5 + 1.5)))
for ax, col, xlabel in [
    (axes[0], 'MAE_deg',       'MAE (°)'),
    (axes[1], 'median_AE_deg', 'Median |error| (°)'),
    (axes[2], 'circ_r',        'Circular r'),
]:
    ax.barh(labels, gabor_summary[col].values, color='steelblue')
    ax.set_xlabel(xlabel)
    ax.axvline(0, color='k', lw=0.8)
    if col == 'MAE_deg':
        ax.axvline(45, color='gray', lw=1, ls='--', label='chance')
        ax.legend(fontsize=8)
fig.suptitle(f'sub-{SUBJECT} {ses_label} — gabor decoding variants', fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# ── detailed plots — best variant ─────────────────────────────────────────────
best_gabor_key = gabor_summary.iloc[0]['variant']
v = gabor_results[best_gabor_key]
results = v['results']
print(f'Best: {best_gabor_key}')
print(f'  MAE={v["mae"]:.1f}°  median|e|={v["median_ae"]:.1f}°  circ_r={v["circ_corr"]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.scatter(results['true_deg'], results['decoded_deg'],
           alpha=0.4, s=14, color='steelblue', linewidths=0)
ax.plot([0, 180], [0, 180], 'k--', lw=1, label='identity')
ax.set_xlabel('True orientation (°)')
ax.set_ylabel('Decoded orientation (°)')
ax.set_title('True vs decoded')
ax.set_xticks([0, 45, 90, 135, 180])
ax.set_yticks([0, 45, 90, 135, 180])
ax.set_xlim(-5, 185); ax.set_ylim(-5, 185)
ax.legend()

ax = axes[1]
ax.hist(results['error_deg'], bins=36, range=(-90, 90),
        color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_xlabel('Circular error (°)')
ax.set_ylabel('Trial count')
ax.set_title(f'Error  (MAE={v["mae"]:.1f}°, r={v["circ_corr"]:.3f})')

label = best_gabor_key.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
fig.suptitle(f'Gabor decoding — {label}', fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# ── posterior width vs accuracy ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(results['circ_std_deg'], results['abs_error_deg'],
           alpha=0.35, s=12, color='steelblue', linewidths=0)
ax.set_xlabel('Posterior width — circular SD (°)')
ax.set_ylabel('Absolute error (°)')
ax.set_title('Posterior width vs accuracy')
r, p = stats.spearmanr(results['circ_std_deg'], results['abs_error_deg'])
print(f'Spearman r = {r:.3f}, p = {p:.3e}')
print(f'Median posterior width: {results["circ_std_deg"].median():.1f}°')
fig.tight_layout()
plt.show()

In [ ]:
# ── 9 example trial posteriors ────────────────────────────────────────────────
pdfs_best    = v['pdfs']
orientations = pdfs_best.columns.values.astype(float)
N_EXAMPLES   = 9
rng          = np.random.default_rng(0)
example_idx  = rng.choice(len(pdfs_best), size=N_EXAMPLES, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(10, 7))
for ax, i in zip(axes.ravel(), example_idx):
    row = pdfs_best.iloc[i].values.astype(float)
    p   = row / np.trapz(row, orientations)
    ax.plot(np.rad2deg(orientations), p, color='steelblue')
    ax.axvline(np.rad2deg(results['true_rad'].iloc[i]),
               color='crimson', lw=1.5, label='true')
    ax.axvline(np.rad2deg(results['decoded_rad'].iloc[i]),
               color='darkorange', lw=1.5, ls='--', label='decoded')
    ax.set_xlim(0, 180)
    ax.set_xlabel('Orientation (°)', fontsize=7)
    ax.set_ylabel('Posterior density', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_title(f'err={results["error_deg"].iloc[i]:+.0f}°', fontsize=8)
handles = [
    plt.Line2D([0], [0], color='crimson',    lw=1.5, label='true'),
    plt.Line2D([0], [0], color='darkorange', lw=1.5, ls='--', label='decoded'),
]
fig.legend(handles=handles, loc='upper right', fontsize=9)
label = best_gabor_key.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
fig.suptitle(f'Example posteriors — {label}', fontsize=10)
fig.tight_layout()
plt.show()

---
## 2 · Abstract value (CHF) decoding

In [ ]:
# ── helper functions (value, linear) ─────────────────────────────────────────

def posterior_mean(pdfs):
    """Compute posterior mean value from a DataFrame of likelihoods.

    pdfs : DataFrame (n_trials × n_grid_values), columns = CHF values.

    Returns dict with decoded (posterior mean), concentration (posterior std),
    and z (raw integral for internal use).
    """
    values    = pdfs.columns.values.astype(float)
    posterior = pdfs.values
    norm      = np.trapz(posterior, values, axis=1)[:, np.newaxis]
    post_norm = posterior / np.where(norm > 0, norm, 1.0)

    decoded   = np.trapz(post_norm * values[np.newaxis, :], values, axis=1)
    post_var  = np.trapz(post_norm * (values[np.newaxis, :] - decoded[:, np.newaxis])**2,
                         values, axis=1)
    post_std  = np.sqrt(post_var.clip(0))
    return dict(decoded=decoded, post_std=post_std)

In [ ]:
# ── discover + load all value decoding variants ───────────────────────────────
value_dir = (Path(BIDS_FOLDER) / 'derivatives' / 'decoding' / 'value'
             / f'sub-{SUBJECT}' / ses_label / 'func')

value_files = sorted(value_dir.glob('*_pars.tsv'))
print(f'Found {len(value_files)} value variant(s):')
for f in value_files:
    print(' ', f.name)

value_results = {}
for tsv in value_files:
    pdfs = pd.read_csv(tsv, sep='\t',
                       index_col=['session', 'run', 'trial_nr', 'true_value_chf'])
    pdfs.columns = pdfs.columns.astype(float)

    stats_ = posterior_mean(pdfs)
    true_val    = pdfs.index.get_level_values('true_value_chf').values.astype(float)
    decoded_val = stats_['decoded']
    error       = decoded_val - true_val
    mae         = np.mean(np.abs(error))
    median_ae   = np.median(np.abs(error))
    r, pval     = stats.pearsonr(true_val, decoded_val)

    res = pd.DataFrame({
        'true_chf':    true_val,
        'decoded_chf': decoded_val,
        'error':       error,
        'abs_error':   np.abs(error),
        'post_std':    stats_['post_std'],
    }, index=pdfs.index)

    value_results[tsv.stem] = dict(
        pdfs=pdfs, results=res,
        mae=mae, median_ae=median_ae, pearson_r=r, pearson_p=pval)
    print(f'  {tsv.stem}:')
    print(f'    MAE={mae:.2f} CHF  median_AE={median_ae:.2f}  r={r:.3f}  p={pval:.3e}')

In [ ]:
# ── comparison table + bar chart ──────────────────────────────────────────────
value_summary = pd.DataFrame([
    dict(variant=k, MAE_chf=v['mae'], median_AE_chf=v['median_ae'],
         pearson_r=v['pearson_r'], pearson_p=v['pearson_p'])
    for k, v in value_results.items()
]).sort_values('MAE_chf')

display(value_summary.set_index('variant').round(3))

labels = [k.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
          for k in value_summary['variant']]

fig, axes = plt.subplots(1, 3, figsize=(14, max(3, len(labels) * 0.5 + 1.5)))
for ax, col, xlabel in [
    (axes[0], 'MAE_chf',      'MAE (CHF)'),
    (axes[1], 'median_AE_chf','Median |error| (CHF)'),
    (axes[2], 'pearson_r',    'Pearson r'),
]:
    ax.barh(labels, value_summary[col].values, color='darkorange')
    ax.set_xlabel(xlabel)
    ax.axvline(0, color='k', lw=0.8)

fig.suptitle(f'sub-{SUBJECT} {ses_label} — value decoding variants', fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# ── detailed plots — best variant ─────────────────────────────────────────────
best_value_key = value_summary.iloc[0]['variant']
v = value_results[best_value_key]
results = v['results']
print(f'Best: {best_value_key}')
print(f'  MAE={v["mae"]:.2f} CHF  r={v["pearson_r"]:.3f}  p={v["pearson_p"]:.3e}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
vmin = min(results['true_chf'].min(), results['decoded_chf'].min())
vmax = max(results['true_chf'].max(), results['decoded_chf'].max())
ax.scatter(results['true_chf'], results['decoded_chf'],
           alpha=0.4, s=14, color='darkorange', linewidths=0)
ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1, label='identity')
ax.set_xlabel('True value (CHF)')
ax.set_ylabel('Decoded value (CHF)')
ax.set_title('True vs decoded')
ax.legend()

ax = axes[1]
ax.hist(results['error'], bins=30, color='darkorange',
        edgecolor='white', linewidth=0.3)
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_xlabel('Error (CHF)')
ax.set_ylabel('Trial count')
ax.set_title(f'Decoding error  (MAE={v["mae"]:.2f} CHF, r={v["pearson_r"]:.3f})')

label = best_value_key.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
fig.suptitle(f'Value decoding — {label}', fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# ── posterior width vs accuracy ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(results['post_std'], results['abs_error'],
           alpha=0.35, s=12, color='darkorange', linewidths=0)
ax.set_xlabel('Posterior SD (CHF)')
ax.set_ylabel('Absolute error (CHF)')
ax.set_title('Posterior width vs accuracy')
r, p = stats.spearmanr(results['post_std'], results['abs_error'])
print(f'Spearman r = {r:.3f}, p = {p:.3e}')
print(f'Median posterior SD: {results["post_std"].median():.2f} CHF')
fig.tight_layout()
plt.show()

In [ ]:
# ── 9 example trial posteriors ────────────────────────────────────────────────
pdfs_best = v['pdfs']
values_grid = pdfs_best.columns.values.astype(float)
N_EXAMPLES  = 9
rng         = np.random.default_rng(0)
example_idx = rng.choice(len(pdfs_best), size=N_EXAMPLES, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(10, 7))
for ax, i in zip(axes.ravel(), example_idx):
    row = pdfs_best.iloc[i].values.astype(float)
    p   = row / np.trapz(row, values_grid)
    ax.plot(values_grid, p, color='darkorange')
    ax.axvline(results['true_chf'].iloc[i],
               color='crimson', lw=1.5, label='true')
    ax.axvline(results['decoded_chf'].iloc[i],
               color='steelblue', lw=1.5, ls='--', label='decoded')
    ax.set_xlabel('Value (CHF)', fontsize=7)
    ax.set_ylabel('Posterior density', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_title(f'err={results["error"].iloc[i]:+.2f} CHF', fontsize=8)
handles = [
    plt.Line2D([0], [0], color='crimson',   lw=1.5, label='true'),
    plt.Line2D([0], [0], color='steelblue', lw=1.5, ls='--', label='decoded'),
]
fig.legend(handles=handles, loc='upper right', fontsize=9)
label = best_value_key.replace(f'sub-{SUBJECT}_{ses_label}_', '').replace('_pars', '')
fig.suptitle(f'Example posteriors — {label}', fontsize=10)
fig.tight_layout()
plt.show()

---
## 3 · Gabor vs value decoding — side-by-side

In [ ]:
# ── mask × noise × smoothing × lambda grid — gabor MAE and value r ───────────
def parse_variant(stem, subject, ses_label):
    """Extract mask, noise, smoothed, lambda flags from output filename stem."""
    s = stem.replace(f'sub-{subject}_{ses_label}_mask-', '')
    parts = s.split('_')
    mask     = parts[0]
    noise    = 'spherical' if 'noise-spherical' in stem else 'full'
    smoothed = 'smoothed' if '_smoothed_' in stem + '_' else 'raw'
    m = re.search(r'_lambda-([0-9.e+\-]+)', stem)
    lambd    = float(m.group(1)) if m else 0.0
    return mask, noise, smoothed, lambd

rows = []
for stem, v in gabor_results.items():
    mask, noise, smoothed, lambd = parse_variant(stem, SUBJECT, ses_label)
    nvox = int(stem.split('nvoxels-')[1].split('_')[0]) if 'nvoxels-' in stem else None
    rows.append(dict(mask=mask, noise=noise, smoothed=smoothed, lambd=lambd, nvoxels=nvox,
                     gabor_MAE=v['mae'], gabor_r=v['circ_corr']))
gabor_df = pd.DataFrame(rows)

rows = []
for stem, v in value_results.items():
    mask, noise, smoothed, lambd = parse_variant(stem, SUBJECT, ses_label)
    nvox = int(stem.split('nvoxels-')[1].split('_')[0]) if 'nvoxels-' in stem else None
    rows.append(dict(mask=mask, noise=noise, smoothed=smoothed, lambd=lambd, nvoxels=nvox,
                     value_MAE=v['mae'], value_r=v['pearson_r']))
value_df = pd.DataFrame(rows)

combined = pd.merge(gabor_df, value_df, on=['mask', 'noise', 'smoothed', 'lambd', 'nvoxels'], how='outer')
combined = combined.sort_values('gabor_MAE')
display(combined.set_index(['mask', 'noise', 'smoothed', 'lambd']).round(3))